# 02 - LangChain y chunking

Objetivo: cargar documentos y dividirlos en chunks razonables antes de vectorizar.


In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv(Path("..").resolve() / ".env")

CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", "650"))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", "100"))
DOCS_DIR = Path("..").resolve() / "docs"

print({
    "DOCS_DIR": str(DOCS_DIR),
    "CHUNK_SIZE": CHUNK_SIZE,
    "CHUNK_OVERLAP": CHUNK_OVERLAP,
})


In [ ]:
loader = DirectoryLoader(
    str(DOCS_DIR),
    glob="**/*.md",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
)
documents = loader.load()
print(f"Documentos cargados: {len(documents)}")
documents[0]


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = splitter.split_documents(documents)
print(f"Chunks generados: {len(chunks)}")

for idx, chunk in enumerate(chunks[:3]):
    chunk.metadata["chunk_id"] = idx
    print("=" * 80)
    print(chunk.metadata.get("source", "sin source"))
    print(chunk.page_content[:400])


In [ ]:
chunk_lengths = [len(chunk.page_content) for chunk in chunks]
print({
    "min": min(chunk_lengths),
    "max": max(chunk_lengths),
    "media_aprox": round(sum(chunk_lengths) / len(chunk_lengths), 1),
})


Prueba sugerida:

1. Cambia `CHUNK_SIZE` o `CHUNK_OVERLAP` en `.env`.
2. Vuelve a ejecutar este notebook.
3. Observa si cambian los trozos que luego irian a embeddings.
